# Automated Tree generation (Generator -> Evaluator)

## Main functions:

In [1]:
# This script uses the generator and evaluator modules to create classification trees.
import json
import time
from generator import generate_structured_questions, AnswerEntry, QuestionEntry
from evaluator import evaluate_structured_questions
from generate_tree import Node
from typing import Optional
from monitor import OpenRouterMonitor

def visualize_tree(node, level=0):
    # Node: question, answers, history, current_classes, children, parent
    if node is not None:
        prefix = "  " * level
        if node.question:
            print(f"{prefix}[L{level}] Q: {node.question}")
            for ans in node.answers:
                if ans in node.children:
                    for child in node.children[ans]:
                        print(f"{prefix}  -> {ans}:")
                        visualize_tree(child, level + 1)
        else:
            print(f"{prefix}[L{level}] Leaf Node: {node.current_classes}")

def generate_tree(parent_node: Node, current_classes: list, past_questions: list, n_questions_to_generate: int, n_questions_to_keep: int, dataset_path: str, provider: str = "openrouter", monitor: Optional[OpenRouterMonitor] = None):
    
    print(f"Generating tree for node: {parent_node.question} with answers {parent_node.answers} with classes: {current_classes}")
    
    
    # recursive function to build the tree
    if len(current_classes) <= 1:
        return [Node(\
            question="",
            answers=[],
            history=parent_node.history,
            current_classes=current_classes)]
    
    # Generate questions using the generator module
    generated_questions = generate_structured_questions(
        current_classes=current_classes,
        past_questions=past_questions,
        number_of_questions=n_questions_to_generate,
        dataset_path=dataset_path,
        provider=provider,
        monitor=monitor
    )
    print(f"Generated {n_questions_to_generate} questions.")
    print(f"Generated Questions: {generated_questions}")
    time.sleep(3) # Sleep for 3 seconds to avoid rate limiting

    # Evaluate the generated questions using the evaluator module
    evaluation_output = evaluate_structured_questions(
        current_classes=current_classes,
        past_questions=past_questions,
        generator_output=generated_questions,
        keep_top_k=n_questions_to_keep,
        dataset_path=dataset_path,
        provider=provider,
        monitor=monitor
    )
    print(f"Evaluated {n_questions_to_generate} questions and kept {n_questions_to_keep}.")
    print(f"Evaluation Output: {evaluation_output}")

    # Extract the top K questions from the evaluation output
    top_questions = evaluation_output.top_k_questions
    
    print(f"Top Questions: {top_questions}")
    # create K trees, one for each top question
    # Create K copies of the current tree(Node), and then assign the top questions to each of them

    # if top questions > n_questions_to_keep, choose one randomly
    if len(top_questions) > n_questions_to_keep:
        top_questions = top_questions[:n_questions_to_keep]
    
    current_nodes = []
    # for each answer in the top questions, create a new node
    for top_question_entry in top_questions:
        
        questionentry: QuestionEntry = top_question_entry.question_entry
        question: str = questionentry.question
        
        answers: list[AnswerEntry] = questionentry.answers # Key-Value pairs of answers e.g. {"Yes": [0, 1, 2], "No": [3, 4, 5]}
        answers = {answer.key: answer.value for answer in answers}

        question_answers = {"question": question, "answers": answers}
        question_answers = json.dumps(question_answers, indent=None)
        
        current_node = Node(
                    question=question,
                    answers=answers.keys(),
                    history=parent_node.history + [question_answers],
                    current_classes=current_classes
                )
        
        # Recursively generate the tree for each answer
        for answer in answers.keys():
            # Filter classes based on the answer
            filtered_classes: list[int] = answers[answer]
            # child_node = Node(
            #     current_classes=filtered_classes,
            # )
            if filtered_classes:
                # Generate the subtree
                children_nodes = generate_tree(
                    current_node,
                    filtered_classes,
                    past_questions + [question_answers],
                    n_questions_to_generate,
                    n_questions_to_keep,
                    dataset_path=dataset_path,
                    provider=provider,
                    monitor=monitor
                    )
                # Add the node to the children of the current node's answers
                # So in the branch of answer1 "Yes", we add multiple children nodes which represent forest for this node
                current_node.children[answer] = children_nodes
                # child_node.parent = current_node
        
        current_nodes.append(current_node)
    return current_nodes

In [2]:
def visualize_tree(node, level=0):
    # Node: question, answers, history, current_classes, children, parent
    # print(node)
    if node is not None:
        prefix = "  " * level
        if node.children != {}:
            print(f"{prefix}[L{level}] Q: {node.question}")
            for ans in node.answers:
                if ans in node.children:
                    if type(node.children[ans]) is list:
                        for child in node.children[ans]:
                            print(f"{prefix}  -> {ans}:")
                            visualize_tree(child, level + 1)
                    else:
                        print(f"{prefix}  -> {ans}:")
                        visualize_tree(node.children[ans], level + 1)
        else:
            class_id = node.current_classes[0] if node.current_classes else None
            if class_id is not None:
                class_name = class_names.get(str(class_id), "Unknown Class")
                print(f"{prefix}[L{level}] Leaf Node: {class_name} (ID: {class_id})")
            else:
                print(f"{prefix}[L{level}] Leaf Node: No classes available")

## CIFAR 10 use case

In [ ]:
import os


current_classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
past_questions = []
n_questions_to_generate = 3
n_questions_to_keep = 1

# Load dataset configuration
with open("dataset_config.json", "r") as f:
    config_data = json.load(f)
    dataset_name = "CIFAR-10"
# dataset_path is used for loading class names and other dataset-specific info
dataset_path = config_data["Dataset_path"]
dataset_path = f'../../{dataset_path}/'
print("Is valid dataset path ?", os.path.exists(dataset_path))
# Create the root node of the tree
root_node_1_root = Node(
    question="",
    answers=[],
    history=[],
    current_classes=current_classes
)


# Generate the tree starting from the root node
current_nodes = generate_tree(
    parent_node=root_node_1_root,
    current_classes=current_classes,
    past_questions=past_questions,
    n_questions_to_generate=n_questions_to_generate,
    n_questions_to_keep=n_questions_to_keep,
    dataset_path=dataset_path,
    provider="openrouter"
)

Generating tree for node:  with answers [] with classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Response JSON: {"question_evaluations":[{"question_entry":{"question":"Is the object in the image a living animal?","explanation":"This question divides man-made vehicles and ships from animals (mammmals, birds, amphibians). Animals are visibly distinct from vehicles and ships by their body shapes, fur, feathers, or skin.","answers":[{"key":"Yes","value":[2,3,4,5,6,7]},{"key":"No","value":[0,1,8,9]}]},"scores":[{"score_name":"Clarity and Visual relevance","score_value":2},{"score_name":"Completeness","score_value":1},{"score_name":"Visual relevance","score_value":3},{"score_name":"Creativity and Logical Grouping","score_value":1},{"score_name":"Maximizing splits","score_value":3},{"score_name":"Overall Score","score_value":10}]},{"question_entry":{"question":"Is the object a type of vehicle or vessel used for transportation?","explanation":"This question separates vehicles and vessels (airplane, aut

In [17]:
current_nodes[0]

Node(question=Is the object in the image a living animal?, answers=dict_keys(['Yes', 'No']), history=['{"question": "Is the object in the image a living animal?", "answers": {"Yes": [2, 3, 4, 5, 6, 7], "No": [0, 1, 8, 9]}}'], current_classes=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], children={'Yes': [Node(question=Is the animal primarily depicted with fur or hair covering its body?, answers=dict_keys(['Yes', 'No']), history=['{"question": "Is the object in the image a living animal?", "answers": {"Yes": [2, 3, 4, 5, 6, 7], "No": [0, 1, 8, 9]}}', '{"question": "Is the animal primarily depicted with fur or hair covering its body?", "answers": {"Yes": [3, 4, 5, 7], "No": [2, 6]}}'], current_classes=[2, 3, 4, 5, 6, 7], children={'Yes': [Node(question=Is the animal typically shown with prominent hooves?, answers=dict_keys(['Yes (hooves)', 'No (paws)']), history=['{"question": "Is the object in the image a living animal?", "answers": {"Yes": [2, 3, 4, 5, 6, 7], "No": [0, 1, 8, 9]}}', '{"question": "Is

In [ ]:
with open("class_names.json", "r") as f:
    class_names = json.load(f)
visualize_tree( current_nodes[0])

[L0] Q: Is the object in the image a living animal?
  -> Yes:
  [L1] Q: Is the animal primarily depicted with fur or hair covering its body?
    -> Yes:
    [L2] Q: Is the animal typically shown with prominent hooves?
      -> Yes (hooves):
      [L3] Q: Does the animal have antlers visible in the image?
        -> Yes (antlers):
        [L4] Leaf Node: deer (ID: 4)
        -> No (no antlers):
        [L4] Leaf Node: horse (ID: 7)
      -> No (paws):
      [L3] Q: Does the animal have a noticeably longer snout or muzzle?
        -> Yes (longer muzzle):
        [L4] Leaf Node: dog (ID: 5)
        -> No (shorter, rounded snout):
        [L4] Leaf Node: cat (ID: 3)
    -> No:
    [L2] Q: Does the animal have feathers covering its body?
      -> Yes:
      [L3] Leaf Node: bird (ID: 2)
      -> No:
      [L3] Leaf Node: frog (ID: 6)
  -> No:
  [L1] Q: Is the object capable of floating on water?
    -> Yes:
    [L2] Leaf Node: ship (ID: 8)
    -> No:
    [L2] Q: Does the object have wings?
 

In [40]:
class TreeNode:
    def __init__(self, question=None, answers=None, message=None, possible_classes=None):
        self.question = question
        self.answers = answers or []
        self.message = message
        self.possible_classes = possible_classes or []
        self.children = {}  # Map answers to child nodes

def build_cifar_tree():
    # ==============================================
    # Animal Subtree (Living Animals)
    # ==============================================
    
    # Fur/Hair animals subtree
    fur_animals = TreeNode(
        question="Is the animal typically shown with prominent hooves?",
        answers=["yes", "no"],
        message="Hooved vs pawed animals"
    )
    
    # Hooved animals
    hooved = fur_animals.children["yes"] = TreeNode(
        question="Does the animal have antlers visible in the image?",
        answers=["yes", "no"],
        message="Antler identification"
    )
    hooved.children["yes"] = TreeNode(message="deer", possible_classes=[4])
    hooved.children["no"] = TreeNode(message="horse", possible_classes=[7])
    
    # Pawed animals
    pawed = fur_animals.children["no"] = TreeNode(
        question="Does the animal have a noticeably longer snout or muzzle?",
        answers=["yes", "no"],
        message="Snout length identification"
    )
    pawed.children["yes"] = TreeNode(message="dog", possible_classes=[5])
    pawed.children["no"] = TreeNode(message="cat", possible_classes=[3])
    
    # Non-fur animals
    non_fur = TreeNode(
        question="Does the animal have feathers covering its body?",
        answers=["yes", "no"],
        message="Feather identification"
    )
    non_fur.children["yes"] = TreeNode(message="bird", possible_classes=[2])
    non_fur.children["no"] = TreeNode(message="frog", possible_classes=[6])
    
    # Main animal branch
    animals = TreeNode(
        question="Is the animal primarily depicted with fur or hair covering its body?",
        answers=["yes", "no"],
        message="Fur/hair identification"
    )
    animals.children["yes"] = fur_animals
    animals.children["no"] = non_fur
    
    # ==============================================
    # Non-Animal Subtree (Objects/Vehicles)
    # ==============================================
    
    # Water-capable objects
    water_objects = TreeNode(message="ship", possible_classes=[8])
    
    # Non-water objects
    non_water = TreeNode(
        question="Does the object have wings?",
        answers=["yes", "no"],
        message="Wing identification"
    )
    non_water.children["yes"] = TreeNode(message="airplane", possible_classes=[0])
    
    # Ground vehicles
    ground_vehicles = non_water.children["no"] = TreeNode(
        question="Is the vehicle larger in size and typically used for transporting goods?",
        answers=["yes", "no"],
        message="Vehicle size and purpose"
    )
    ground_vehicles.children["yes"] = TreeNode(message="truck", possible_classes=[9])
    ground_vehicles.children["no"] = TreeNode(message="automobile", possible_classes=[1])
    
    # Non-animal branch
    non_animals = TreeNode(
        question="Is the object capable of floating on water?",
        answers=["yes", "no"],
        message="Water capability"
    )
    non_animals.children["yes"] = water_objects
    non_animals.children["no"] = non_water
    
    # ==============================================
    # Main Tree Structure
    # ==============================================
    root = TreeNode(
        question="Is the object in the image a living animal?",
        answers=["yes", "no"],
        message="Root classification - Animal vs Non-animal"
    )
    
    root.children["yes"] = animals
    root.children["no"] = non_animals
    
    return root

def print_tree(node, level=0):
    if node is not None:
        prefix = "  " * level
        if node.question:
            print(f"{prefix}[L{level}] Q: {node.question}")
            for ans in node.answers:
                if ans in node.children:
                    print(f"{prefix}  -> {ans}:")
                    print_tree(node.children[ans], level+1)
        else:
            print(f"{prefix}[L{level}] Leaf Node: {node.message} (ID: {node.possible_classes[0] if node.possible_classes else 'None'})")

if __name__ == "__main__":
    print("CIFAR-10 Classification Tree:")
    tree = build_cifar_tree()
    print_tree(tree)

CIFAR-10 Classification Tree:
[L0] Q: Is the object in the image a living animal?
  -> yes:
  [L1] Q: Is the animal primarily depicted with fur or hair covering its body?
    -> yes:
    [L2] Q: Is the animal typically shown with prominent hooves?
      -> yes:
      [L3] Q: Does the animal have antlers visible in the image?
        -> yes:
        [L4] Leaf Node: deer (ID: 4)
        -> no:
        [L4] Leaf Node: horse (ID: 7)
      -> no:
      [L3] Q: Does the animal have a noticeably longer snout or muzzle?
        -> yes:
        [L4] Leaf Node: dog (ID: 5)
        -> no:
        [L4] Leaf Node: cat (ID: 3)
    -> no:
    [L2] Q: Does the animal have feathers covering its body?
      -> yes:
      [L3] Leaf Node: bird (ID: 2)
      -> no:
      [L3] Leaf Node: frog (ID: 6)
  -> no:
  [L1] Q: Is the object capable of floating on water?
    -> yes:
    [L2] Leaf Node: ship (ID: 8)
    -> no:
    [L2] Q: Does the object have wings?
      -> yes:
      [L3] Leaf Node: airplane (ID: 0

## Same process but for MSCOCO dataset

In [ ]:
import numpy as np
import sys

current_classes = np.arange(1, 91).tolist()
past_questions = []
n_questions_to_generate = 3
n_questions_to_keep = 1
dataset_name = "MSCOCO"

# Redirect stdout to a file
sys.stdout = open("console_output.txt", "w", buffering=1)
monitor = OpenRouterMonitor()

# load configuration to get the class names
with open('../../config.json', 'r') as f:
    config_data = json.load(f)
    config_data = config_data[dataset_name]

# dataset_path is used for loading class names and other dataset-specific info
dataset_path = config_data["Dataset_path"]
dataset_path = f'../../{dataset_path}/'

# Create the root node of the tree
root_node_1_root = Node(
    question="",
    answers=[],
    history=[],
    current_classes=current_classes
)

# Generate the tree starting from the root node
current_nodes = generate_tree(
    parent_node=root_node_1_root,
    current_classes=current_classes,
    past_questions=past_questions,
    n_questions_to_generate=n_questions_to_generate,
    n_questions_to_keep=n_questions_to_keep,
    dataset_path=dataset_path,
    provider="openrouter",
    monitor=monitor
)

In [4]:
with open("../../datasets/MSCOCO/class_names.json", "r") as f:
    class_names = json.load(f)
visualize_tree( current_nodes[0])

NameError: name 'current_nodes' is not defined